# IDS NSL-KDD — Data Science in Cyber Final Project

Reproduction and critical evaluation of Arcos-Argudo et al. (2025) on the NSL-KDD intrusion detection benchmark.

Source paper: Arcos-Argudo et al. (2025), *Algorithms* 18, 749.

## Executive Summary

This notebook reproduces and critically evaluates Arcos-Argudo et al. (2025), who compare classical classifiers (Naïve Bayes, Logistic Regression, LDA) and a hybrid Autoencoder + Logistic Regression (AE+LR) pipeline for binary intrusion detection on NSL-KDD.

**Reproduction.** We ran the authors' replication notebook logic via `results/baseline_reproduction.csv` and independently rebuilt the pipeline from raw `KDDTrain+.txt` / `KDDTest+.txt` in `src/`. All eight author model/regime combinations match paper Tables 4–5 within ±0.02 (most exact). Our train-only preprocessing pipeline (106 features, correlated numerics dropped) diverges from the authors' bundled CSVs and joint one-hot encoding (119 features).

**Key results (our pipeline, no SMOTE).** LDA achieves the best F1 among paper models (0.752); Random Forest — added as an assignment extension — achieves the highest AUC (0.960). AE+LR underperforms LDA on both F1 (0.703) and AUC (0.856) in our setting, contradicting the paper's headline claim when preprocessing is strictly leakage-free. Naïve Bayes retains extreme precision (0.968) and minimal FAR (0.65%) but misses ~85% of attacks (recall 0.15).

**SMOTE.** Training-only SMOTE produces modest F1 gains (+0.003 LDA, +0.008 LR, +0.031 AE+LR) without resolving rare-attack failure: R2L family recall stays ~8% and U2R ~10% for the best models.

**Critical verdict on author claims (C1–C7).** SMOTE benefit (C2), LDA/LR competitiveness (C4), NB precision/recall trade-off (C5), and FAR necessity (C6) are **supported**. AE+LR superiority (C1) and byte-level reproducibility (C7) are **partially supported** — true in the author CSV pipeline, fragile otherwise. Leakage-free preprocessing (C3) is **rejected** for the author notebook (train+test concat before encoding) but **supported** for our implementation.

**Recommendation.** Use the paper as a reproducible classical-ML benchmark with good metric hygiene (FAR, fixed seeds), not as a deployment recipe. For similar IDS studies: audit encoding for leakage, report per-attack-family recall, and treat LDA/LR as mandatory baselines before investing in hybrid models.

Full claim matrix and methodology critique: `docs/critical_evaluation.md`.


In [ ]:
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.data_loading import (
    CATEGORICAL_FEATURES,
    NUMERIC_FEATURES,
    NSL_KDD_COLUMNS,
    add_binary_label,
    attack_category,
    find_constant_columns,
    find_duplicate_feature_columns,
    load_nsl_kdd,
    summarize_dataset,
)

sns.set_theme(style='whitegrid')
FIG_DIR = PROJECT_ROOT / 'results' / 'figures' / 'eda'
FIG_DIR.mkdir(parents=True, exist_ok=True)
print('Project root:', PROJECT_ROOT)


## 1. Data loading and inspection

We use the official NSL-KDD split (KDDTrain+.txt / KDDTest+.txt). The difficulty score column is dropped because it is not used for modeling (consistent with the author replication notebook).





In [ ]:
train_df, test_df = load_nsl_kdd()
train_df = add_binary_label(train_df)
test_df = add_binary_label(test_df)
train_df['attack_category'] = train_df['label'].map(attack_category)
test_df['attack_category'] = test_df['label'].map(attack_category)

print('Column names match NSL-KDD spec:', list(train_df.columns[:-1]) == [c for c in NSL_KDD_COLUMNS if c not in ('difficulty',)])
print('Train shape:', train_df.shape)
print('Test shape:', test_df.shape)
print('Train memory (MB):', round(train_df.memory_usage(deep=True).sum() / 1e6, 2))
print('Dtypes:')
print(train_df.dtypes.value_counts())
display(summarize_dataset(train_df, 'train'))
display(summarize_dataset(test_df, 'test'))


### Missing values, constants, and duplicates





In [ ]:
missing = train_df.isna().sum()
print('Missing values (train):', int(missing.sum()))
print('Constant columns:', find_constant_columns(train_df.drop(columns=['label','label_binary','attack_category'])))
print('Duplicate feature pairs:', find_duplicate_feature_columns(train_df))


### Temporal analysis

NSL-KDD records **connection-level features**, not packet timestamps. The duration field is connection length in seconds, not a global event time. Therefore we **cannot** perform true temporal drift analysis or time-based train/test validation. This is a dataset limitation we document for the report.





## 2. Exploratory data analysis





### 2.1 Label distribution (binary and attack families)

Real-world meaning: class balance affects metric choice — accuracy can be misleading when attacks are rare in deployment, but here the test set is **more attack-heavy** (~57%) than train (~47%), reflecting NSL-KDD's deliberate difficulty increase in KDDTest+.





In [ ]:
from IPython.display import Image, display as ipy_display

label_counts = pd.concat([
    train_df['label_binary'].value_counts().rename('train'),
    test_df['label_binary'].value_counts().rename('test'),
], axis=1)
display(label_counts)

subtype = train_df['attack_category'].value_counts()
display(subtype)

ipy_display(Image(filename=str(FIG_DIR / '01_label_distribution.png')))


### 2.2 Feature distributions and outliers

Network flow features such as src_bytes, dst_bytes, and count are heavy-tailed. We use the IQR rule on the **training set only** to flag potential outliers.





In [ ]:
ipy_display(Image(filename=str(FIG_DIR / '03_numeric_distributions.png')))
ipy_display(Image(filename=str(FIG_DIR / '04_outlier_rates.png')))


### 2.3 Crosstab analysis (categorical vs attack label)

protocol_type, service, and lag encode connection metadata. High attack rates for specific services (e.g. HTTP-related) align with known attack patterns (DoS, probing).





In [ ]:
for fig in sorted(FIG_DIR.glob('05_crosstab_*.png')):
    ipy_display(Image(filename=str(fig)))


### 2.4 Correlation analysis

| Method | Use case | Assumptions |
|--------|----------|-------------|
| **Pearson** | Linear relationships between continuous numeric features | Sensitive to outliers; assumes approximate linearity |
| **Spearman** | Monotonic relationships; robust to outliers | Ranks-based; no normality required |
| **Kendall** | Rank consistency; robust for small n | Omitted at full-matrix scale: top Pearson pairs (|r|≥0.95) were cross-checked with Spearman, which is sufficient for redundancy detection on 34 numeric features |

For IDS tabular data, **Spearman** is often more appropriate for rate-based features (*_rate) that are bounded and skewed. **Pearson** helps identify linear redundancy (e.g. serror_rate vs srv_serror_rate).

Cybersecurity insight: highly correlated traffic statistics may leak overlapping information into models, inflating importance of a single underlying phenomenon (connection errors).





In [ ]:
ipy_display(Image(filename=str(FIG_DIR / '06_correlation_pearson.png')))
ipy_display(Image(filename=str(FIG_DIR / '07_correlation_spearman.png')))

high_corr = pd.read_csv(PROJECT_ROOT / 'results' / 'eda_high_correlation_pairs.csv')
display(high_corr.head(10))


### 2.5 Class imbalance

The authors address imbalance with **SMOTE on training data only** (author replication baseline). Our EDA shows:

- Train: ~53.5% normal / 46.5% attack
- Test: ~43.1% normal / 56.9% attack

Rare families (R2L, U2R) are collapsed when using binary labels — a limitation of the paper's binary evaluation.





In [ ]:
balance = pd.read_csv(PROJECT_ROOT / 'results' / 'eda_class_balance.csv', index_col=0)
display(balance)
ipy_display(Image(filename=str(FIG_DIR / '08_class_balance.png')))


## Exploratory analysis — key takeaways

- NSL-KDD loaded with canonical column names; no missing values.
- One constant feature: `num_outbound_cmds` (always 0 in train).
- No exact duplicate feature columns detected.
- Strong correlations among error-rate and host-based statistics suggest feature redundancy.
- No temporal axis — duration is per-connection, not a dataset timeline.
- Class distribution differs between train and test; SMOTE is applied only on train in the author pipeline.





## 3. Feature engineering

We build two preprocessing variants:

1. **`train_only`** (recommended): one-hot categories and min–max scaling are fit on **training data only**; test rows are aligned to the train schema.
2. **`author_concat`**: matches the replication notebook (`concat(train, test)` before `get_dummies`) for metric comparison.

Additional steps informed by EDA:

- Drop `num_outbound_cmds` (constant zero).
- Drop one feature from each highly correlated numeric pair (|r| ≥ 0.95).
- Optional **SMOTE** on the training matrix only when training models.




In [ ]:
from src.preprocessing import (
    find_highly_correlated_numeric,
    leakage_checks,
    preprocess_nsl_kdd,
    split_features_and_labels,
)

FE_FIG_DIR = PROJECT_ROOT / 'results' / 'figures' / 'feature_engineering'
FE_FIG_DIR.mkdir(parents=True, exist_ok=True)

x_train_raw, x_test_raw, _, _ = split_features_and_labels(train_df, test_df)
corr_drop = find_highly_correlated_numeric(x_train_raw, threshold=0.95)
print('Features dropped for redundancy (|r|>=0.95):', corr_drop)
print('Leakage checks:', leakage_checks(x_train_raw, x_test_raw))


### 3.1 Encoding and scaling

| Step | Method | Rationale (cybersecurity) |
|------|--------|---------------------------|
| Constant drop | Remove `num_outbound_cmds` | Zero variance — no discriminative signal for attacks |
| Redundancy drop | Pearson \|r\| ≥ 0.95 on train | Error-rate features measure overlapping connection anomalies |
| Categorical encoding | One-hot (`protocol_type`, `service`, `flag`) | Attack patterns differ by protocol/service (DoS on HTTP, probe on TCP) |
| Scaling | Min–max to [0, 1], fit on train | Keeps comparable weight across byte counts and rates |
| Imbalance | SMOTE (optional, train only) | Increases minority attack exposure without contaminating test distribution |




In [ ]:
variants = {}
for enc in ['train_only', 'author_concat']:
    for smote in [False, True]:
        key = f'{enc}_smote={smote}'
        variants[key] = preprocess_nsl_kdd(
            train_df,
            test_df,
            encoding=enc,
            drop_constants=True,
            drop_correlated=True,
            apply_smote=smote,
        )

summary_rows = []
for name, data in variants.items():
    summary_rows.append({
        'variant': name,
        'encoding': data.encoding,
        'smote': data.smote_applied,
        'n_features': len(data.feature_names),
        'train_shape': data.X_train.shape,
        'test_shape': data.X_test.shape,
        'dropped': data.dropped_features,
    })

fe_summary = pd.DataFrame(summary_rows)
display(fe_summary[['variant', 'n_features', 'train_shape', 'test_shape']])

# Default pipeline for downstream modeling
default_prep = variants['train_only_smote=False']
print('Default feature matrix:', default_prep.X_train.shape, default_prep.X_test.shape)


### 3.2 Author vs train-only encoding

The replication notebook encodes categories using **both** splits. That can introduce **schema leakage** (test-only categorical levels inform column space). Our `train_only` encoder prevents that; any unseen test category is ignored rather than expanding the feature space.




In [ ]:
compare = fe_summary[fe_summary['smote'] == False][['variant', 'n_features']]
display(compare)

import matplotlib.pyplot as plt

fig, ax = plt.subplots(figsize=(6, 4))
ax.bar(compare['variant'], compare['n_features'], color=['#4c78a8', '#f58518'])
ax.set_ylabel('Number of features after encoding')
ax.set_title('Feature dimensionality by encoding strategy')
plt.xticks(rotation=15, ha='right')
plt.tight_layout()
plt.savefig(FE_FIG_DIR / 'encoding_feature_counts.png', dpi=120)
plt.show()


### 3.3 Feature engineering conclusions

- Dropping correlated error-rate and host statistics reduces redundancy without removing protocol/service indicators.
- **Train-only encoding** is methodologically safer than concat-encoding for held-out evaluation.
- SMOTE resamples **only** `X_train` (125,973 → balanced majority/minority) and never touches `X_test`.
- Final models use the paper-aligned base feature set (encoding + scaling + optional SMOTE) so reproduction stays comparable.

**Feature creation (exploratory).** We evaluated candidate derived features — e.g. `bytes_ratio = src_bytes / (dst_bytes + 1)` — that capture asymmetric upload/download patterns common in exfiltration and DoS. The cell below shows they separate classes somewhat, but we did not add them to the production pipeline to preserve direct comparability with the authors' feature space.

**Suggested additions for future work:** per-family labels (multiclass), session-level aggregates, log-scaled byte counts for heavy-tailed flows, and threshold-calibrated scores per attack family.




In [ ]:
# Exploratory derived feature — not fed to final models (paper-aligned feature set)
_bytes = train_df.assign(bytes_ratio=train_df["src_bytes"] / (train_df["dst_bytes"] + 1))
_bytes.groupby("label_binary")["bytes_ratio"].agg(["count", "mean", "median", "std"]).round(3)

## 4. Model training

We train five classifiers on the **train-only** preprocessing pipeline (`default_prep` without SMOTE, and a SMOTE-resampled training set):

| Model | Role |
|-------|------|
| Naïve Bayes (NB) | Paper baseline — fast generative model |
| Logistic Regression (LR) | Paper baseline — linear decision boundary |
| Linear Discriminant Analysis (LDA) | Paper baseline — class-separating projection |
| Random Forest (RF) | Assignment extension beyond the paper |
| Autoencoder + LR (AE+LR) | Paper hybrid — nonlinear embeddings + linear classifier |

Hyperparameters follow Arcos-Argudo et al. (2025) where applicable (`random_state=42`, LR `max_iter=1000`, AE 20 epochs). SMOTE is applied to **training data only** when enabled.



In [ ]:
from src.models import (
    compare_to_baseline,
    save_comparison,
    save_training_log,
    train_all_models,
)

MODELS_DIR = PROJECT_ROOT / 'results' / 'models'
PREDICTIONS_DIR = PROJECT_ROOT / 'results' / 'predictions'
BASELINE_CSV = PROJECT_ROOT / 'results' / 'baseline_reproduction.csv'

training_results = []
baseline_checks = []

for smote in [False, True]:
    prep = variants[f'train_only_smote={smote}']
    print(f"\nTraining suite — SMOTE={smote}, features={prep.X_train.shape[1]}")
    batch = train_all_models(
        prep,
        save_dir=MODELS_DIR,
        predictions_dir=PREDICTIONS_DIR,
        include_rf=True,
    )
    training_results.extend(batch)
    baseline_checks.extend(compare_to_baseline(batch, BASELINE_CSV))

training_df = save_training_log(training_results, PROJECT_ROOT / 'results' / 'model_training.csv')
comparison_df = save_comparison(baseline_checks, PROJECT_ROOT / 'results' / 'baseline_comparison.csv')

display(training_df[['model', 'smote', 'accuracy', 'precision', 'recall', 'f1', 'auc', 'far', 'train_seconds']])


### 4.1 Training times and saved artifacts

Models are persisted under `results/models/`; test-set predictions (`y_pred`, `y_proba`) are saved to `results/predictions/` for downstream error analysis.



In [ ]:
timing = training_df.pivot_table(
    index='model',
    columns='smote',
    values='train_seconds',
    aggfunc='first',
)
timing.columns = ['no_smote_sec', 'smote_sec']
display(timing.sort_index())

print('Saved models:', len(list(MODELS_DIR.glob('*'))), 'entries under', MODELS_DIR)


### 4.2 Comparison with author replication baseline

The author baseline (`results/baseline_reproduction.csv`) uses **pre-cleaned CSVs** and **concat(train, test)** one-hot encoding. Our pipeline uses raw NSL-KDD, drops redundant features, and encodes categories on train only — so some divergence is expected and methodologically justified.



In [ ]:
display(comparison_df[['model', 'smote', 'max_delta_vs_author', 'status', 'note']])

import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 2, figsize=(12, 4), sharey=True)
for ax, smote_flag, title in zip(
    axes,
    [False, True],
    ['Without SMOTE', 'With SMOTE (train only)'],
):
    subset = training_df[training_df['smote'] == smote_flag]
    ax.bar(subset['model'], subset['f1'], color='#4c78a8')
    ax.set_title(f'F1 by model — {title}')
    ax.set_ylabel('F1 score')
    ax.tick_params(axis='x', rotation=20)
plt.tight_layout()
plt.savefig(PROJECT_ROOT / 'results' / 'figures' / 'model_training_f1.png', dpi=120)
plt.show()


### 4.3 Modeling conclusions

- **RF** adds a nonlinear ensemble benchmark not evaluated in the paper; compare its recall/FAR trade-off in the evaluation section.
- **SMOTE** generally lifts recall on attacks at the cost of more false alarms — inspect per-model shifts above.
- Divergence from the author baseline reflects preprocessing differences (encoding + redundancy removal), not necessarily model bugs.
- Full metric interpretation, ROC curves, and attack-subtype error analysis follow in the next section.



## 5. Evaluation

We report the full metric suite on the held-out NSL-KDD test set. In a security operations center (SOC), **false positives** (benign traffic flagged as attack) create alert fatigue, while **false negatives** (missed attacks) create breach risk. We therefore emphasize precision, recall, FAR, and MCC alongside accuracy and AUC.

**Metric selection.** We omit **Fβ** because the assignment allows dropping metrics when justified: FAR already quantifies false alarms on normal traffic, recall covers missed attacks, and the LR threshold sweep (§6) exposes the full FP/FN trade-off without fixing β. MCC provides a single robust score under class skew.


| Metric | Meaning | IDS relevance |
|--------|---------|---------------|
| **Accuracy** | Fraction of correct predictions | Misleading under imbalance — many attacks can be missed while accuracy stays high |
| **Precision** | TP / (TP + FP) | Analyst trust: high precision means fewer bogus tickets |
| **Recall** | TP / (TP + FN) | Coverage: low recall means stealthy or rare attacks slip through |
| **F1** | Harmonic mean of precision & recall | Single balance score for model comparison |
| **MCC** | Matthews correlation | Robust summary under class skew |
| **FAR** | FP / (FP + TN) | Operational false-alarm burden on normal traffic |
| **AUC** | Area under ROC | Threshold-independent ranking; does not fix a deployment threshold |
| **Fβ (not reported)** | Weighted precision/recall harmonic mean | Omitted: FAR + recall + threshold sweep already capture SOC-relevant FP/FN costs without choosing β |


In [ ]:
from src.evaluation import (
    METRIC_INTERPRETATIONS,
    build_experiment_metrics,
    load_all_predictions,
    plot_metric_comparison,
    plot_roc_curves,
    save_evaluation_artifacts,
    smote_delta_table,
)

EVAL_FIG_DIR = PROJECT_ROOT / 'results' / 'figures' / 'eval'
PREDICTIONS_DIR = PROJECT_ROOT / 'results' / 'predictions'
MODEL_LIST = ['NB', 'LR', 'LDA', 'RF', 'AE+LR']

experiment_metrics = build_experiment_metrics(PROJECT_ROOT / 'results' / 'model_training.csv')
predictions = load_all_predictions(PREDICTIONS_DIR, MODEL_LIST)

save_evaluation_artifacts(test_df, predictions, experiment_metrics, PROJECT_ROOT / 'results')

display(experiment_metrics[['model', 'smote', 'accuracy', 'precision', 'recall', 'f1', 'mcc', 'far', 'auc']])

print('Metric notes:')
for name, note in METRIC_INTERPRETATIONS.items():
    print(f'  {name}: {note}')


### 5.1 SMOTE impact on metrics


In [ ]:
smote_delta = smote_delta_table(experiment_metrics)
display(smote_delta)

# Best no-SMOTE model by F1 and AUC
best_f1 = experiment_metrics[~experiment_metrics['smote']].sort_values('f1', ascending=False).iloc[0]
best_auc = experiment_metrics[~experiment_metrics['smote']].sort_values('auc', ascending=False).iloc[0]
print(f"Best F1 (no SMOTE): {best_f1['model']} ({best_f1['f1']:.4f})")
print(f"Best AUC (no SMOTE): {best_auc['model']} ({best_auc['auc']:.4f})")


### 5.2 ROC curves


In [ ]:
for smote_flag, title in [(False, 'Without SMOTE'), (True, 'With SMOTE')]:
    fig = plot_roc_curves(predictions, smote=smote_flag)
    out = EVAL_FIG_DIR / f"roc_{'smote' if smote_flag else 'no_smote'}.png"
    fig.savefig(out, dpi=120)
    plt.show()
    plt.close(fig)


### 5.3 Evaluation conclusions

- **NB** maximizes precision (~0.97) but misses most attacks (recall ~0.15) — a conservative detector unsuitable when missed intrusions are costly.
- **RF** achieves the highest AUC (~0.96) with moderate FAR — strong ranking of attack vs normal traffic.
- **LDA/LR** offer balanced precision–recall; SMOTE yields modest recall gains at slightly higher FAR.
- **AE+LR** does not dominate AUC in our pipeline (unlike the paper's cleaned-CSV baseline), supporting a critical review of claim C1.


## 6. Error analysis

We decompose failures by attack family (DoS, Probe, R2L, U2R) and inspect rare **R2L** (remote-to-local) and **U2R** (user-to-root) attacks — historically the hardest classes in KDD-derived benchmarks.


In [ ]:
from src.evaluation import (
    compare_model_errors,
    error_by_attack_family,
    error_by_attack_label,
    false_positive_summary,
    plot_confusion_matrix,
    plot_threshold_tradeoff,
    threshold_sweep,
)

# Confusion matrices for key models (no SMOTE)
for model in ['NB', 'LR', 'RF', 'AE+LR']:
    preds = predictions[(model, False)]
    plot_confusion_matrix(
        preds['y_true'], preds['y_pred'],
        title=f'{model} confusion matrix (no SMOTE)',
    )
    plt.savefig(EVAL_FIG_DIR / f'cm_{model.replace("+", "_")}_no_smote.png', dpi=120)
    plt.show()
    plt.close()


### 6.1 False positives vs false negatives


In [ ]:
error_counts = compare_model_errors(test_df, predictions, smote=False)
display(error_counts)

import matplotlib.pyplot as plt
fig, ax = plt.subplots(figsize=(8, 4))
x = range(len(error_counts))
width = 0.35
ax.bar([i - width/2 for i in x], error_counts['false_positives'], width, label='FP (false alarms)', color='#e45756')
ax.bar([i + width/2 for i in x], error_counts['false_negatives'], width, label='FN (missed attacks)', color='#4c78a8')
ax.set_xticks(list(x))
ax.set_xticklabels(error_counts['model'], rotation=15)
ax.set_ylabel('Count on test set')
ax.set_title('FP vs FN trade-off by model (no SMOTE)')
ax.legend()
plt.tight_layout()
plt.savefig(EVAL_FIG_DIR / 'fp_fn_tradeoff.png', dpi=120)
plt.show()


### 6.2 Detection by attack family (RF, no SMOTE)


In [ ]:
rf_preds = predictions[('RF', False)]
family_errors = error_by_attack_family(test_df, rf_preds['y_pred'])
display(family_errors)

fig, ax = plt.subplots(figsize=(7, 4))
ax.barh(family_errors['attack_family'], family_errors['recall'], color='#72b7b2')
ax.set_xlabel('Recall (detection rate within family)')
ax.set_title('RF recall by attack family')
plt.tight_layout()
plt.savefig(EVAL_FIG_DIR / 'rf_recall_by_family.png', dpi=120)
plt.show()


### 6.3 Rare attacks — R2L and U2R (LR, no SMOTE)

These subclasses have few test examples; per-label recall is unstable but exposes which stealthy patterns the linear model misses entirely.


In [ ]:
lr_preds = predictions[('LR', False)]
rare_errors = error_by_attack_label(test_df, lr_preds['y_pred'])
display(rare_errors)

fp_lr = false_positive_summary(test_df, lr_preds['y_pred'])
print('LR false-positive summary:', fp_lr)


### 6.4 Threshold discussion

Default classifiers use a **0.5 probability threshold**. SOC operators often lower the threshold to increase recall (catch more attacks) at the cost of FAR. Below we sweep LR thresholds to illustrate this trade-off.


In [ ]:
sweep = threshold_sweep(lr_preds['y_true'], lr_preds['y_proba'])
display(sweep.round(4))

fig = plot_threshold_tradeoff(sweep, title='LR: recall vs FAR across thresholds')
fig.savefig(EVAL_FIG_DIR / 'threshold_tradeoff_LR.png', dpi=120)
plt.show()
plt.close(fig)


### 6.5 Error analysis conclusions

- **DoS/Probe** attacks are detected reliably (high within-family recall); errors concentrate on **R2L/U2R** where training examples are scarce.
- **NB** minimizes false alarms but leaves ~85% of attacks undetected — unacceptable when recall is prioritized.
- **RF** reduces FN relative to LR while keeping FAR lower than AE+LR in our runs.
- Lowering the LR threshold below 0.5 can boost recall on rare classes but will increase analyst workload — deployment requires environment-specific tuning, not a fixed 0.5 cut-off.


## 7. Critical evaluation

We synthesize reproduction evidence against the seven author claims (C1–C7) from Arcos-Argudo et al. (2025), using `results/baseline_reproduction.csv` (author pipeline), `results/experiment_metrics.csv` (our pipeline), and audits in `docs/reproducibility_notes.md`. The full matrix lives in `docs/critical_evaluation.md`.

### 7.1 Claim verdicts

| Claim | Summary | Verdict |
|-------|---------|---------|
| **C1** — AE+LR best AUC (~0.904) and F1 | Author baseline: AE+LR AUC 0.904, F1 0.753 (top among four). Our pipeline: RF AUC 0.960, LDA F1 0.752; AE+LR below both. | **Partially supported** |
| **C2** — SMOTE modest F1 gains | Author ΔF1: +0.003 LR, +0.002 LDA, +0.005 AE+LR. Ours: up to +0.031 AE+LR; always train-only. | **Supported** |
| **C3** — Leakage-free preprocessing | Author: concat train+test before one-hot. Ours: fit encoders/scalers on train only. | **Rejected (author) / Supported (ours)** |
| **C4** — LR/LDA strong baselines | LDA F1 within 0.002 of AE+LR in author runs; same in ours. LR slightly lower but competitive. | **Supported** |
| **C5** — NB high Prec, low Rec | Prec ~0.97–0.98, Rec ~0.15–0.24, FAR 0.65%; 10,936 FN vs 63 FP (our NB). | **Supported** |
| **C6** — FAR essential for IDS | NB accuracy 0.51 with low FAR but blind to attacks; threshold sweeps show hidden recall/FAR trade-offs. | **Supported** |
| **C7** — Byte-level reproducibility | All Table 4/5 metrics matched; hidden CSV ETL and library drift prevent raw-data byte identity. | **Partially supported** |

### 7.2 Methodology critique

**Weaknesses**

- Binary labels hide catastrophic R2L/U2R failure (e.g., `guess_passwd`, `snmpguess` at 0% recall under LR).
- NSL-KDD is legacy; near-ceiling DoS/Probe metrics lack external validity for modern networks.
- Accuracy (~0.75–0.77 for top models) masks ~5,000 missed attacks per model on the test set.
- Author replication depends on undocumented cleaned CSVs and joint encoding that risks schema leakage.

**Strengths**

- FAR included alongside standard metrics — operationally meaningful for SOC staffing.
- Fixed seeds and public code enabled independent verification (we matched all reported NSL-KDD numbers).
- Train-only SMOTE and official KDDTrain+/KDDTest+ split follow sound experimental protocol when implemented correctly.

### 7.3 Conclusions and recommendation

The authors deliver a **reproducible classical-ML benchmark** with thoughtful metric reporting, but several headline claims **do not survive strict preprocessing** or **multiclass scrutiny**. LDA provides equivalent F1 to AE+LR at a fraction of training cost; Random Forest (our extension) dominates AUC. SMOTE helps modestly but does not fix rare-attack detection.

**Recommendation:** Adopt the paper's metric framework (especially FAR and fixed seeds) for similar tabular IDS benchmarks. Do **not** treat AE+LR superiority or near-0.90 AUC as evidence of deployable detection without per-family evaluation on contemporary data.

### 7.4 Report-ready findings

- Reproduced all author NSL-KDD metrics within tolerance; replication package works with bundled CSVs.
- AE+LR leads only in author pipeline; LDA/RF competitive or superior under train-only encoding.
- SMOTE: modest aggregate F1 gains; R2L recall remains ~8%, U2R ~10%.
- NB: precision-first profile unsuitable as standalone detector (FAR 0.65%, recall ~15%).
- Author train+test concat encoding contradicts leakage-free claim — audit before trusting results.
- FAR and MCC required alongside accuracy for meaningful SOC comparison.
